In [1]:
import pandas as pd
import torch

In [2]:
df = pd.read_csv("IMDB_Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [8]:
df.shape

(49582, 2)

# Pre-Processing

## 1. Converting To Lower Case

In [9]:
df["review"] = df["review"].str.lower()

In [10]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


## 2. Removing the URLs

In [11]:
import re

In [12]:
def remove_urls(text): # (pattern,replace,string)
    return re.sub(r"http\S+", "", text) # eg:- (https:/www.google.com)

df["review"] = df["review"].apply(remove_urls)

In [13]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production. <br /><br />the...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically there's a family where a little boy ...,negative
4,"petter mattei's ""love in the time of money"" is...",positive


## 3. Removing Punctuations

In [14]:
def remove_punctuations(text): # (pattern,replace,string)
    return re.sub(r"[^A-Za-z0-9\s]", "", text) 

df["review"] = df["review"].apply(remove_punctuations)

In [15]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


## 4. Removing HTML

In [16]:
def remove_html(text): # (pattern,replace,string)
    return re.sub(r"<.*?>", "", text) 

df["review"] = df["review"].apply(remove_html)

In [17]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


## 5. Removing stopwords

In [18]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mdfaz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mdfaz\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mdfaz\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [20]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    tokens = word_tokenize(text)

    filtered_words = [
        word for word in tokens
        if word not in stop_words
    ]

    return " ".join(filtered_words)

df["review"] = df["review"].apply(remove_stopwords)

In [21]:
df.head()

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production br br filming tech...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


## 6. Stemming

In [22]:
# running -> run
# played -> play
# Porter Stemminng

from nltk.stem import PorterStemmer

In [23]:
ps  = PorterStemmer()

def stemming(text):
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [24]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod youll hoo...,positive
1,wonder littl product br br film techniqu unass...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic there famili littl boy jake think there ...,negative
4,petter mattei love time money visual stun film...,positive


### 7. Encoding

In [25]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])


In [26]:
y = df["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [28]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4108221 stored elements and shape (49582, 5000)>

## Data & Data Loaders

In [29]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [30]:
print(X_train.shape)
print(X_test.shape)

(39665, 5000)
(9917, 5000)


In [31]:
import torch
from torch.utils.data import TensorDataset , DataLoader

In [32]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [33]:
train_set = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.float32)
)

test_set = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test.values, dtype=torch.float32)
)

In [34]:
train_loader = DataLoader(train_set,shuffle=True,batch_size=64)
test_loader = DataLoader(test_set,shuffle=True,batch_size=64)

## Build Our RNN

In [35]:
import torch.nn as nn
import torch.optim as optim

In [36]:
class RNN(nn.Module):
    def __init__(self,input_size,hidden_size=128,num_layer=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layer = num_layer

        # RNN Layer
        self.rnn = nn.RNN(input_size,hidden_size,num_layer,batch_first=True)

        ## fully connected layer (Many -> one archoitevcture)
        self.fc = nn.Linear(hidden_size,1)

    def forward(self,x):
        # optional step => (num of layers , batch size  , hidden size )
        h0 = torch.zeros(
            self.num_layer,
            x.size(0),
            self.hidden_size,
            device=x.device # traferring to gpu 
    )

        out,_ = self.rnn (x,h0)
        # 1st val = hidden state of all the timesteps => (batch,seq_len,hidden size )
        # 2nd val = final hidden state of last timestep

        out = self.fc(out[:,-1,:])
        return out
        

### Build Model 

In [37]:
# transferring to run on gpu 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
#model = model.to(device)

Using device: cuda


In [38]:
input_size = X.shape[1] 

#model = RNN(input_size).to(device)
model = RNN(input_size, hidden_size=128, num_layer=1).to(device)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Traing Our RNN Model

In [39]:
# unsqueeze , squeeze

In [40]:
epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss=0

    for Xb , yb in train_loader:
        
        optimizer.zero_grad() 

        # Move data to GPU(next 2 lines)
        Xb = Xb.to(device)
        yb = yb.to(device)

        
        Xb = Xb.unsqueeze(1) # add singleton direction
        outputs = model(Xb)  # (batch_size,1)
        outputs = torch.sigmoid(outputs.squeeze())    # (batch_size,) => probabilty

        loss = criterion(outputs,yb) # compute loss

        loss.backward() # back.. prop..

        optimizer.step() # Weights update
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"epoch ={epoch+1}/{epochs} and loss = {avg_loss}")
            
        

epoch =1/10 and loss = 0.34400265620600795
epoch =2/10 and loss = 0.23325802242803959
epoch =3/10 and loss = 0.2172310983581889
epoch =4/10 and loss = 0.20982543735374365
epoch =5/10 and loss = 0.20566497759953623
epoch =6/10 and loss = 0.20263158332916997
epoch =7/10 and loss = 0.20035034394672802
epoch =8/10 and loss = 0.19815749570487007
epoch =9/10 and loss = 0.19662978957377134
epoch =10/10 and loss = 0.19568485890425022


## Evaluation

In [41]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals=0
    total_test_loss = 0

    for Xb,yb in test_loader:
        # Move data to GPU(next 2 lines)
        Xb = Xb.to(device)
        yb = yb.to(device)
        
        Xb=Xb.unsqueeze(1)
        outputs = model(Xb)
        outputs = torch.sigmoid(outputs.squeeze())
        
         # Calculate loss
        loss = criterion(outputs, yb)
        total_test_loss += loss.item()
        
        # Calculate accuracy
        predicted = (outputs > 0.5).float()
    
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    # Average test loss
    avg_test_loss = total_test_loss / len(test_loader)

    # Accuracy
    accuracy = correct_vals / tot_vals * 100
    
    print(f"Test Loss = {avg_test_loss:.4f}")    
    print(f"accuracy = {accuracy:.4f}")


Test Loss = 0.3193
accuracy = 87.2542
